# Descarga y preparación de datos agrícolas

## Proyecto Agronomía

Esta libreta documenta la obtención y preparación de los datos agrícolas
utilizados para estudiar los factores asociados con el éxito de la
producción agrícola en los municipios de México entre **2003 y 2025**.

Los archivos agrícolas se obtienen anualmente de la fuente de datos de
la Secretaría de Agricultura y posteriormente se integran en un único
conjunto histórico.

Debido a que la estructura de los archivos presenta algunos cambios a
lo largo del periodo de estudio, el procesamiento incluye una etapa de
normalización para mantener un esquema consistente entre los años.

La lógica de descarga y transformación se encuentra implementada en el
paquete `proyecto_agronomia`. Esta libreta permite ejecutar, inspeccionar
y validar el proceso de manera reproducible.

### Periodo de estudio

**2003–2025**

### Flujo de trabajo

1. Descargar los archivos agrícolas anuales.
2. Verificar la disponibilidad de los archivos.
3. Inspeccionar un archivo agrícola.
4. Normalizar las diferencias históricas del esquema.
5. Construir el conjunto agrícola histórico.
6. Generar variables derivadas e indicadores de calidad.
7. Validar el conjunto resultante.

In [1]:
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIO_INICIAL,
    ANIO_FINAL,
    ANIOS,
)

from proyecto_agronomia.dataset import (
    descargar_datos_agricolas,
)

from proyecto_agronomia.features import (
    cargar_archivo_agricola,
    construir_historico_agricola,
    generar_features_agricolas,
)

2026-10-03 17:14:09.421 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## 1. Descarga de los datos agrícolas

La fuente agrícola proporciona un archivo independiente para cada año
del periodo de estudio.

Para construir la serie histórica se consideran los años comprendidos
entre **2003 y 2025**.

Los archivos originales se almacenan en:

`data/raw/agricultura/`

Los datos de esta carpeta representan la información original obtenida
de la fuente y no son modificados directamente durante las etapas
posteriores del procesamiento.

La función de descarga verifica si cada archivo ya existe localmente.
Cuando un archivo está disponible, se reutiliza en lugar de volver a
descargarlo.

In [2]:
print(
    f"Periodo de estudio: "
    f"{ANIO_INICIAL}-{ANIO_FINAL}"
)

print(
    f"Años considerados: "
    f"{len(ANIOS)}"
)

print(
    f"Directorio de datos originales: "
    f"{AGRICULTURA_RAW_DIR}"
)

Periodo de estudio: 2003-2025
Años considerados: 23
Directorio de datos originales: B:\MCD2\JW\agro\proyecto_agronomia\data\raw\agricultura


## 2. Ejecución de la descarga

Se utiliza la función `descargar_datos_agricolas()` del módulo
`proyecto_agronomia.dataset`.

La función recorre los años del periodo de estudio y conserva los
archivos que ya se encuentran disponibles localmente, por lo que la
operación puede ejecutarse nuevamente sin descargar innecesariamente
los mismos datos.

In [3]:
archivos_agricolas = descargar_datos_agricolas(
    ANIOS
)

print(
    f"Archivos disponibles: "
    f"{len(archivos_agricolas)}"
)

Datos agrícolas:  65%|██████▌   | 15/23 [00:00<00:00, 143.74it/s]

2026-10-03 17:14:10.179 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2003] El archivo ya existe.
2026-10-03 17:14:10.187 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2004] El archivo ya existe.
2026-10-03 17:14:10.193 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2005] El archivo ya existe.
2026-10-03 17:14:10.198 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2006] El archivo ya existe.
2026-10-03 17:14:10.205 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2007] El archivo ya existe.
2026-10-03 17:14:10.210 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2008] El archivo ya existe.
2026-10-03 17:14:10.215 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2009] El archivo ya existe.
2026-10-03 17:14:10.220 | INFO     | proy

Datos agrícolas:  65%|██████▌   | 15/23 [00:00<00:00, 143.74it/s]

2026-10-03 17:14:10.313 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2021] El archivo ya existe.
2026-10-03 17:14:10.325 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2022] El archivo ya existe.
2026-10-03 17:14:10.335 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2023] El archivo ya existe.
2026-10-03 17:14:10.343 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2024] El archivo ya existe.
2026-10-03 17:14:10.352 | INFO     | proyecto_agronomia.datasets.agricultura:descargar_datos_agricolas:47 - [2025] El archivo ya existe.


Datos agrícolas: 100%|██████████| 23/23 [00:00<00:00, 126.15it/s]

Archivos disponibles: 23


## 3. Verificación de los archivos

Antes de iniciar el procesamiento se verifica que exista un archivo
para cada uno de los años considerados en el estudio.

Esta validación permite detectar años faltantes antes de construir el
conjunto histórico.

In [4]:
disponibilidad = pd.DataFrame(
    {
        "Anio": ANIOS,
        "Archivo": [
            f"agricola_{anio}.csv"
            for anio in ANIOS
        ],
        "Disponible": [
            (
                AGRICULTURA_RAW_DIR
                / f"agricola_{anio}.csv"
            ).exists()
            for anio in ANIOS
        ],
    }
)

disponibilidad

,Anio,Archivo,Disponible
0,2003,agricola_2003.csv,True
1,2004,agricola_2004.csv,True
2,2005,agricola_2005.csv,True
3,2006,agricola_2006.csv,True
4,2007,agricola_2007.csv,True
5,2008,agricola_2008.csv,True
6,2009,agricola_2009.csv,True
7,2010,agricola_2010.csv,True
8,2011,agricola_2011.csv,True
9,2012,agricola_2012.csv,True


In [5]:
print(
    f"Archivos esperados: "
    f"{len(ANIOS)}"
)

print(
    f"Archivos disponibles: "
    f"{disponibilidad['Disponible'].sum()}"
)

print(
    f"Archivos faltantes: "
    f"{(~disponibilidad['Disponible']).sum()}"
)

Archivos esperados: 23
Archivos disponibles: 23
Archivos faltantes: 0


## 4. Inspección de los datos agrícolas

Cada archivo contiene registros agrícolas asociados con una entidad,
municipio, cultivo, ciclo productivo y modalidad.

Entre las principales variables disponibles se encuentran:

- superficie sembrada;
- superficie cosechada;
- superficie siniestrada;
- volumen de producción;
- rendimiento;
- precio medio rural;
- valor de la producción.

A continuación se inspecciona uno de los archivos del periodo de
estudio antes de construir el conjunto histórico.

In [6]:
ruta_ejemplo = (
    AGRICULTURA_RAW_DIR
    / "agricola_2025.csv"
)

agricultura_ejemplo = cargar_archivo_agricola(
    ruta_ejemplo,
    2025,
)

agricultura_ejemplo.head()

2026-10-03 17:14:10.718 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2025] Columnas normalizadas.


,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1500.0,1500.0,0.0,35250.0,23.50,654.00,23053500.00
1,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5670000,Brócoli,50.0,50.0,0.0,850.0,17.00,6887.78,5854613.00
2,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5490000,Avena forrajera en verde,165.0,165.0,0.0,3700.0,22.42,645.00,2386500.00
3,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5670000,Brócoli,60.0,60.0,0.0,785.2,13.09,5161.69,4052958.99
4,2025,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,2,...,Tonelada,5740000,Calabacita,30.0,30.0,0.0,490.0,16.33,5285.21,2589752.90


In [7]:
print(
    f"Observaciones: "
    f"{len(agricultura_ejemplo):,}"
)

print(
    f"Variables: "
    f"{agricultura_ejemplo.shape[1]}"
)

print(
    f"Estados: "
    f"{agricultura_ejemplo['Idestado'].nunique()}"
)

print(
    f"Municipios: "
    f"{agricultura_ejemplo[['Idestado', 'Idmunicipio']].drop_duplicates().shape[0]:,}"
)

print(
    f"Cultivos: "
    f"{agricultura_ejemplo['Nomcultivo'].nunique():,}"
)

Observaciones: 35,902
Variables: 24
Estados: 32
Municipios: 2,441
Cultivos: 314


## 5. Normalización histórica

Los archivos agrícolas no mantienen exactamente el mismo esquema de
columnas durante todo el periodo 2003–2025.

Se identificaron cambios históricos en variables como el nombre del
cultivo y el precio medio rural.

Para construir una serie consistente, el pipeline normaliza estas
diferencias antes de concatenar los archivos anuales.

Entre los cambios considerados se encuentran:

- `Nomcultivo Sin Um` → `Nomcultivo`
- `Precio` → `Preciomediorural`

También se convierten las principales variables cuantitativas a formato
numérico y se eliminan separadores de miles cuando están presentes.

Los archivos originales permanecen sin modificaciones en
`data/raw/agricultura/`; la normalización se realiza durante la
construcción del conjunto histórico.

## 6. Construcción del conjunto agrícola histórico

Una vez disponibles y normalizados los archivos anuales, se integran
los datos correspondientes al periodo **2003–2025** en un único
conjunto histórico.

Durante este proceso:

1. Se carga cada archivo anual.
2. Se normalizan las diferencias de nombres entre años.
3. Se convierten las principales variables cuantitativas a formato
   numérico.
4. Se concatenan todos los registros conservando el año y las
   características originales de cada observación.

Esta integración permite trabajar con todo el periodo de estudio bajo
un esquema común.

In [8]:
agricultura_historica = construir_historico_agricola()

agricultura_historica.shape

Procesando datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

2026-10-03 17:14:11.423 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2003] Columnas normalizadas.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:15,  1.40it/s]

2026-10-03 17:14:11.912 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2003] 35,667 registros procesados.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:15,  1.40it/s]

...

Procesando datos agrícolas:  96%|█████████▌| 22/23 [00:14<00:00,  1.59it/s]

2026-10-03 17:14:25.253 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:61 - [2025] Columnas normalizadas.


Procesando datos agrícolas: 100%|██████████| 23/23 [00:14<00:00,  1.59it/s]

2026-10-03 17:14:25.686 | INFO     | proyecto_agronomia.features:construir_historico_agricola:147 - [2025] 35,902 registros procesados.


(825089, 24)

In [9]:
agricultura_historica.head()

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Nomunidad,Idcultivo,Nomcultivo,Sembrada,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion
0,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5490000,Avena forrajera en verde,1062.0,1062.0,0.0,28529.0,26.86,323.4,9226278.6
1,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5900000,Cebada forrajera en verde,2.0,2.0,0.0,54.0,27.00,250.0,13500.0
2,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,5980000,Centeno forrajero en verde,17.0,17.0,0.0,1020.0,60.00,350.0,357000.0
3,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,9090000,Triticale forrajero en verde,24.0,24.0,0.0,720.0,30.00,400.0,288000.0
4,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,Tonelada,15050000,Pastos y praderas,415.0,415.0,0.0,25830.0,62.24,314.9,8133867.0


In [10]:
print(
    f"Observaciones totales: "
    f"{len(agricultura_historica):,}"
)

print(
    f"Variables: "
    f"{agricultura_historica.shape[1]}"
)

print(
    f"Periodo: "
    f"{agricultura_historica['Anio'].min()}-"
    f"{agricultura_historica['Anio'].max()}"
)

print(
    f"Años disponibles: "
    f"{agricultura_historica['Anio'].nunique()}"
)

Observaciones totales: 825,089
Variables: 24
Periodo: 2003-2025
Años disponibles: 23


## 7. Cobertura temporal

Se verifica el número de registros disponibles para cada año del
periodo de estudio.

Esta revisión permite comprobar que los 23 años están representados y
detectar cambios importantes en la cantidad de observaciones entre
periodos.

In [11]:
registros_por_anio = (
    agricultura_historica
    .groupby("Anio")
    .size()
    .reset_index(name="Registros")
)

registros_por_anio

,Anio,Registros
0,2003,35667
1,2004,36204
2,2005,36520
3,2006,36056
4,2007,36136
5,2008,35785
6,2009,35562
7,2010,36228
8,2011,36299
9,2012,36263


## 8. Revisión de calidad de los datos

Antes de generar variables derivadas se realizan comprobaciones básicas
sobre las variables agrícolas cuantitativas.

Se revisan:

- valores faltantes;
- registros duplicados;
- valores negativos;
- superficie sembrada igual a cero;
- superficie cosechada mayor que la sembrada;
- balance entre superficie cosechada, siniestrada y sembrada.

Las observaciones potencialmente inconsistentes no se eliminan
automáticamente. En etapas posteriores se generan indicadores que
permiten identificarlas y conservar el dato original.

In [12]:
columnas_numericas = [
    "Sembrada",
    "Cosechada",
    "Siniestrada",
    "Volumenproduccion",
    "Rendimiento",
    "Preciomediorural",
    "Valorproduccion",
]

faltantes = (
    agricultura_historica[columnas_numericas]
    .isna()
    .sum()
    .to_frame("Faltantes")
)

faltantes

,Faltantes
Sembrada,0
Cosechada,0
Siniestrada,0
Volumenproduccion,0
Rendimiento,875
Preciomediorural,873
Valorproduccion,0


In [13]:
duplicados_exactos = (
    agricultura_historica
    .duplicated()
    .sum()
)

print(
    f"Registros duplicados exactos: "
    f"{duplicados_exactos:,}"
)

Registros duplicados exactos: 0


In [14]:
negativos = (
    agricultura_historica[columnas_numericas]
    .lt(0)
    .sum()
    .to_frame("Valores_negativos")
)

negativos

,Valores_negativos
Sembrada,0
Cosechada,0
Siniestrada,0
Volumenproduccion,0
Rendimiento,0
Preciomediorural,0
Valorproduccion,0


## 9. Generación de variables derivadas

A partir de las variables agrícolas originales se genera
`TasaCosecha`, definida como:

\[
\text{TasaCosecha} =
\frac{\text{Cosechada}}{\text{Sembrada}}
\]

Esta variable representa la proporción de la superficie sembrada que
llegó a ser cosechada.

Cuando `Sembrada = 0`, la tasa se mantiene como valor faltante para
evitar una división indefinida.

También se generan indicadores de calidad que permiten identificar
observaciones potencialmente inconsistentes sin modificar ni eliminar
los registros originales:

- `FlagSembradaCero`
- `FlagCosechadaMayorSembrada`
- `FlagBalanceSuperficie`

In [15]:
agricultura_features = generar_features_agricolas(
    agricultura_historica
)

agricultura_features.head()

,Anio,Idestado,Nomestado,Idddr,Nomddr,Idcader,Nomcader,Idmunicipio,Nommunicipio,Idciclo,...,Cosechada,Siniestrada,Volumenproduccion,Rendimiento,Preciomediorural,Valorproduccion,TasaCosecha,FlagSembradaCero,FlagCosechadaMayorSembrada,FlagBalanceSuperficie
0,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,1062.0,0.0,28529.0,26.86,323.4,9226278.6,1.0,False,False,False
1,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,2.0,0.0,54.0,27.00,250.0,13500.0,1.0,False,False,False
2,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,17.0,0.0,1020.0,60.00,350.0,357000.0,1.0,False,False,False
3,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,24.0,0.0,720.0,30.00,400.0,288000.0,1.0,False,False,False
4,2003,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,Aguascalientes,1,...,415.0,0.0,25830.0,62.24,314.9,8133867.0,1.0,False,False,False


In [16]:
agricultura_features.shape

(825089, 28)

In [17]:
print(
    "TasaCosecha faltante:",
    agricultura_features[
        "TasaCosecha"
    ].isna().sum()
)

print(
    "Sembrada = 0:",
    agricultura_features[
        "FlagSembradaCero"
    ].sum()
)

print(
    "Cosechada > Sembrada:",
    agricultura_features[
        "FlagCosechadaMayorSembrada"
    ].sum()
)

print(
    "Cosechada + Siniestrada > Sembrada:",
    agricultura_features[
        "FlagBalanceSuperficie"
    ].sum()
)

TasaCosecha faltante: 60
Sembrada = 0: 60
Cosechada > Sembrada: 2
Cosechada + Siniestrada > Sembrada: 125


In [18]:
tasa_mayor_uno = (
    agricultura_features[
        agricultura_features["TasaCosecha"] > 1
    ][
        [
            "Anio",
            "Nomestado",
            "Nommunicipio",
            "Nomcultivo",
            "Sembrada",
            "Cosechada",
            "TasaCosecha",
        ]
    ]
)

tasa_mayor_uno

,Anio,Nomestado,Nommunicipio,Nomcultivo,Sembrada,Cosechada,TasaCosecha
810046,2025,Oaxaca,San José Estancia Grande,Pastos y praderas,1062.80,1063.14,1.000320
810212,2025,Oaxaca,Santiago Tapextla,Pastos y praderas,1205.02,1206.63,1.001336


## Resultado

Se construyó un conjunto agrícola histórico homogéneo para el periodo
2003–2025.

El conjunto conserva los registros originales y agrega variables
derivadas necesarias para las siguientes etapas del análisis.

La variable `TasaCosecha` permitirá estudiar la proporción de superficie
sembrada que llega a cosecharse y posteriormente analizar su relación
con factores agrícolas, geográficos y climáticos.

Las observaciones potencialmente inconsistentes se conservaron y fueron
identificadas mediante indicadores de calidad, evitando realizar
correcciones o eliminaciones automáticas sin una justificación
metodológica.

El conjunto resultante queda preparado para su posterior integración
con los datos climáticos mediante las variables de año, estado y
municipio.